# Task 2: Coronavirus Tweet Sentiment Classification

By default, the notebook runs both feature representations and both classifiers. The trailing comma is required when selecting only one item.

In [ ]:
FEATURE_EXTRACTORS_TO_RUN = ('tfidf', 'glove')
MODELS_TO_RUN = ('logistic_regression', 'linear_svm')

In a fresh Colab runtime, the next cells clone the public project repository into `/content/machine-learning-large-assignment`.

The pipeline downloads all external data directly into the temporary runtime:

- [Training CSV](https://huggingface.co/datasets/ImpactTom6819/covid_tweets_practice/resolve/5e441292c9120525a4ddd54eba3d65187d66bf49/Corona_NLP_train.csv)
- [Test CSV](https://huggingface.co/datasets/ImpactTom6819/covid_tweets_practice/resolve/5e441292c9120525a4ddd54eba3d65187d66bf49/Corona_NLP_test.csv)
- [GloVe Twitter 100-dimensional embeddings](https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-twitter-100/glove-twitter-100.gz)
- [Original Kaggle dataset page](https://www.kaggle.com/datasets/datatattle/covid-19-nlp-text-classification)

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/MinhTrinhh/machine-learning-large-assignment.git'
REPO_BRANCH = 'main'
COLAB_ROOT = Path('/content/machine-learning-large-assignment')

Clone the public repository into Colab's temporary storage if it is not already present in this runtime.

In [ ]:
if not COLAB_ROOT.exists():
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
        REPO_URL, str(COLAB_ROOT),
    ])

PROJECT_ROOT = COLAB_ROOT

Validate the checkout and add its root directory to Python's import path.

In [ ]:
if not (PROJECT_ROOT / 'modules').is_dir():
    raise FileNotFoundError(
        f'Project modules were not found under {PROJECT_ROOT}. '
        'Confirm that the public GitHub repository and branch are available.'
    )

PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Install all libraries declared in `requirements.txt`, then import the supporting modules automatically.

In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--disable-pip-version-check', '-r',
    str(PROJECT_ROOT / 'requirements.txt'),
])
print('Dependencies installed successfully.')

Load pandas and the shared plotting module used throughout the notebook.

In [ ]:
import importlib
import pandas as pd
import modules.plotter as plotter

plotter = importlib.reload(plotter)

Load the reusable modules for downloading, preparation, features, training, and evaluation.

In [ ]:
from modules.data_downloader import load_dataset
from modules.data_preprocessor import prepare_data, save_preparation_audit
from modules.evaluator import evaluate_models, validate_models
from modules.feature_extractor import GloveFeatureExtractor, TfidfFeatureExtractor
from modules.models import LinearSVMModel, LogisticRegressionModel
from modules.text_eda import run_eda
from modules.trainer import (
    PreparedData,
    fit_final_models,
    load_precomputed_feature_representations,
    save_feature_representations,
    save_precomputed_feature_representations,
    tune_models,
)

print('Pipeline modules imported successfully.')

## 1. Exploratory data analysis

EDA uses the official training split. The test file is inspected only for schema and integrity checks. The download function caches the verified CSV files under the runtime's project-level `data/` directory.

In [ ]:
eda = run_eda(PROJECT_ROOT)
display(eda['quality'])
display(eda['class_distribution'])

## 2. Prepare shared data

Remove duplicates and train/test overlap, normalize the text, and create one stratified train-validation split shared by every configured model.

In [ ]:
train_raw, test_raw = load_dataset(PROJECT_ROOT / 'data')
train, validation, test, audit = prepare_data(train_raw, test_raw)
prepared_data = PreparedData(train, validation, test, audit)
save_preparation_audit(
    audit,
    PROJECT_ROOT / 'outputs/generated/preprocess/data_preparation_audit.json',
)
prepared_data.audit

## 3. Visualize extracted features before modelling

Fit visualization-only copies of the configured feature extractors on the clean training corpus, then display PCA/SVD, t-SNE, and UMAP projections. The same stratified 5,000-tweet sample is used for every plot.

In [ ]:
# Reload plotting code when this section is rerun in an existing kernel.
plotter = importlib.reload(plotter)

visualization_extractors = {
    'tfidf': TfidfFeatureExtractor(),
    'glove': GloveFeatureExtractor(PROJECT_ROOT),
}
unknown_features = set(FEATURE_EXTRACTORS_TO_RUN) - visualization_extractors.keys()
if unknown_features:
    raise ValueError(f'Unknown feature selection: {sorted(unknown_features)}')

Fit each selected extractor and transform the same complete training corpus.

In [ ]:
visualization_representations = {}
full_train = prepared_data.full_train
for feature_name in FEATURE_EXTRACTORS_TO_RUN:
    extractor = visualization_extractors[feature_name]
    extractor.fit(full_train['ProcessedTweet'])
    visualization_representations[feature_name] = (
        extractor.transform(full_train['ProcessedTweet']),
        extractor.label,
    )

Create the PCA/SVD, t-SNE, and UMAP projections and display their metadata.

In [ ]:
feature_space_metadata = plotter.plot_feature_space_visualizations(
    PROJECT_ROOT,
    representations=visualization_representations,
    labels=full_train['Sentiment'],
)
display(feature_space_metadata)

## 4. Configure the controlled experiments

All configured models use the same prepared data. Hyperparameters are selected without using the official test labels, and final evaluation is performed on the untouched test split.

### 4.1 Validate the experiment selection

Define the available classifiers and stop early if a configuration name is invalid.

In [ ]:
AVAILABLE_FEATURE_EXTRACTORS = {'tfidf', 'glove'}
MODEL_FACTORIES = {
    'logistic_regression': LogisticRegressionModel,
    'linear_svm': LinearSVMModel,
}

if not FEATURE_EXTRACTORS_TO_RUN or not MODELS_TO_RUN:
    raise ValueError('Select at least one feature extractor and one model')

unknown_features = set(FEATURE_EXTRACTORS_TO_RUN) - AVAILABLE_FEATURE_EXTRACTORS
unknown_models = set(MODELS_TO_RUN) - MODEL_FACTORIES.keys()
if unknown_features:
    raise ValueError(f'Unknown feature selection: {sorted(unknown_features)}')
if unknown_models:
    raise ValueError(f'Unknown model selection: {sorted(unknown_models)}')

### 4.2 Build fresh classifier instances

Create the selected models and use the smaller predefined logistic-regression grid for dense GloVe features.

In [ ]:
def build_models(feature_name):
    models = [MODEL_FACTORIES[name]() for name in MODELS_TO_RUN]
    if feature_name == 'glove' and 'logistic_regression' in MODELS_TO_RUN:
        index = MODELS_TO_RUN.index('logistic_regression')
        models[index] = LogisticRegressionModel((0.5, 1.0, 2.0))
    return models

### 4.3 Prepare reusable feature matrices

GloVe matrices are saved and reloaded before modelling; TF-IDF remains inside its fitted scikit-learn pipelines.

In [ ]:
def prepare_feature_inputs(extractor):
    if extractor.name != 'glove':
        return None

    save_precomputed_feature_representations(
        extractor,
        prepared_data.train,
        prepared_data.validation,
        prepared_data.full_train,
        prepared_data.test,
        PROJECT_ROOT,
    )
    inputs = load_precomputed_feature_representations(extractor, PROJECT_ROOT)
    print('Reloaded saved GloVe .npy files as classifier inputs.')
    return inputs

### 4.4 Tune and validate models

Run cross-validation on the development data, then choose a model using the shared validation split.

In [ ]:
def tune_and_validate(extractor, models, inputs):
    development_features = None if inputs is None else inputs['development']
    validation_features = None if inputs is None else inputs['validation']

    tuning = tune_models(
        extractor, models, prepared_data.train, PROJECT_ROOT,
        train_features=development_features,
    )
    validation = validate_models(
        tuning, prepared_data.validation,
        validation_features=validation_features,
    )
    return tuning, validation

### 4.5 Fit final models and save features

Refit the tuned models on all training data and save the selected feature representation.

In [ ]:
def fit_and_save(extractor, tuning, validation, inputs):
    full_train_features = None if inputs is None else inputs['train']
    training = fit_final_models(
        tuning, prepared_data.full_train, PROJECT_ROOT,
        full_train_features=full_train_features,
    )

    if inputs is None:
        save_feature_representations(
            training,
            prepared_data.full_train,
            prepared_data.test,
            validation.selected_model,
            PROJECT_ROOT,
        )
    return training

### 4.6 Evaluate on the official test split

Evaluate the fitted models once on the untouched test data.

In [ ]:
def evaluate_experiment(extractor, training, validation, inputs):
    test_features = None if inputs is None else inputs['test']
    return evaluate_models(
        training, extractor, prepared_data.test, validation, PROJECT_ROOT,
        test_features=test_features,
    )

### 4.7 Combine the experiment steps

This small coordinator applies the same workflow to either feature extractor.

In [ ]:
def run_experiment(extractor):
    models = build_models(extractor.name)
    inputs = prepare_feature_inputs(extractor)
    tuning, validation = tune_and_validate(extractor, models, inputs)
    training = fit_and_save(extractor, tuning, validation, inputs)
    return evaluate_experiment(extractor, training, validation, inputs)

### 4.8 Run TF-IDF experiments

Fit sparse word-level TF-IDF features and train the selected classifiers.

In [ ]:
results = {}
if 'tfidf' in FEATURE_EXTRACTORS_TO_RUN:
    results['tfidf'] = run_experiment(TfidfFeatureExtractor())

### 4.9 Run GloVe-Twitter-100 experiments

Mean-pool pretrained Twitter embeddings and save the development, validation, complete-training, and test matrices as `.npy` files before model training. The first run downloads and extracts the vectors into the runtime's project cache.

In [ ]:
if 'glove' in FEATURE_EXTRACTORS_TO_RUN:
    results['glove'] = run_experiment(GloveFeatureExtractor(PROJECT_ROOT))

## 5. Results

Display each feature-model metric table and confusion matrix, then create the consolidated comparison chart.

In [ ]:
for feature_name, result in results.items():
    plotter.plot_experiment_results(
        result, prepared_data.test['Sentiment'], PROJECT_ROOT
    )
    print(f"{result['feature_label']} selected: {result['selected_model']}")
    display(result['test_metrics'])

Combine the test metrics into one table and generate the final comparison chart.

In [ ]:
combined_test_metrics = pd.concat(
    [result['test_metrics'] for result in results.values()],
    ignore_index=True,
)
plotter.plot_model_comparison(
    combined_test_metrics,
    PROJECT_ROOT / 'outputs/graphics/comparison/feature_model_comparison.png',
)
display(combined_test_metrics)

## Generated artifacts

All artifacts are generated inside the Colab runtime.\
Feature representations and aligned labels are stored under `features/`\
Figures, metrics, audits, and fitted models are stored under `outputs/`.

In [ ]:
if combined_test_metrics.empty:
    raise RuntimeError('The experiment completed without producing test metrics.')

print('Feature matrices:', PROJECT_ROOT / 'features')
print('Figures, metrics, and models:', PROJECT_ROOT / 'outputs')
print('Runtime -> Run all completed successfully.')